# OpenAI API 中的 Skills

将一个 CSV 分析工作流打包成技能（skill），上传后在托管 shell 中使用 GPT-6 Astra 运行它。你也可以将相同的文件提供给本地 shell 运行时。完整参考请参阅 [Skills 文档](https://developers.openai.com/api/docs/guides/tools-skills)。

## 什么是技能？

技能是一组可复用的文件（指令 + 脚本 + 资源），以文件夹形式打包，并以一个必需的 `SKILL.md` 清单文件为核心。OpenAI 会将该文件包复制到执行环境中，使模型能够按需读取指令并运行代码。

在托管 shell 中，当你将技能附加到 shell 工具环境（`environment.type="container_auto"`）时，会发生以下过程：


- 服务将技能上传并解压到运行时中
- 服务读取 `SKILL.md` 的 frontmatter（name/description），然后将每个技能的 `name`、`description` 和 `path` 添加到用户提示上下文中，让模型知道该技能的存在
- 如果模型决定调用某个技能，它会使用 `path` 读取 `SKILL.md`，然后通过 shell 工具浏览文件并执行脚本

技能指令与其他用户提供的指令具有相同的优先级。

技能适用于流程：即"_如何做_"很重要的可重复工作流（步骤、分支逻辑、格式规则、脚本）。当你希望自己的流程具备以下特点时，技能就很有用：

- 在不同提示/智能体之间复用
- 独立进行版本管理和发布
- 只在需要时调用（而不是写死在每个系统提示中）

### 何时使用技能


**在以下情况下，技能尤其合适且强大……**

1. **你需要一组可复用、可独立进行版本管理的行为。**
示例："PowerPoint 格式化流程"、"公司专用报告生成器"、"标准数据清洗管道"。
2. **你的工作流条件分支很多，或者像复杂的流程图一样层层分支。**
示例：如果 X → 做这个；否则如果 Y → 做那个；再加上验证 + 重试。
3. **你的工作流需要代码执行和本地产物。**
任何能从脚本、模板、测试夹具或参考资源中受益、且这些资源应与指令放在一起的场景。技能本身就被设计为这些资源的 zip 包。
4. **你希望保持系统提示精简。**
将稳定的流程放入技能；系统提示只保留全局行为。
5. **多个智能体或团队共享同一种"内部风格"。**
技能是一种很好的"组织标准库"模式。
6. **你需要可复现性**
技能天然支持通过技能版本进行版本固定（参见下方的版本管理部分）。


**在以下情况下，技能就不那么理想……**

- 这确实是一个**一次性**任务（在对话中写一个快速的内联脚本就可以了）。
- 你主要需要的是**实时外部数据或副作用**。（那是工具/API 调用的职责。）
- 流程每天都在变（技能在工作流稳定之后才最能发挥作用）。


## 技能 vs. 工具 vs. 系统提示

当边界不清晰时，系统提示和工具 schema 就会变得臃肿。将三者结合使用，可以保持条理清晰，并帮助模型表现得更好。下面是一个简单的框架：

**系统提示：全局行为和约束**

适用于：
- 安全边界、语气、拒绝方式
- 每一轮都适用的"始终做 X"原则
- 小而稳定的策略

避免：

- 把冗长的多步骤流程放在这里（这会让每一轮都变得臃肿，而且容易出问题）



**工具："在外部世界中做某事"**

当模型必须完成以下操作时，使用工具：

- 调用外部服务或数据库
- 产生副作用（环境之外的任务，例如取消订单或发送电子邮件）
- 获取实时状态

工具应当：

- 范围窄而明确
- 具有强类型的输入
- 明确说明副作用

**技能：打包好的流程（+ 代码 + 资源）**

当你希望模型完成以下事情时，使用技能：

- 遵循可重复的工作流
- 使用脚本/模板
- 在沙盒中执行代码
- 只在某些时候执行，而不是每次都执行


## 技能打包：SKILL.md 与文件夹布局

### 文件夹结构

技能就是一个文件夹包。示例如下：
- `SKILL.md`（必需）
- 脚本，例如 `*.py`、`*.js`（可选）
- 辅助文件和 `requirements.txt`
- 资源、模板、示例输入


### SKILL.md frontmatter

OpenAI 模型期望名称和描述来自 frontmatter（这对发现和路由很重要）。请将 name 和 description 放在 `SKILL.md` 的 frontmatter 中。每次 API `create` 调用上传一个技能包（一个顶层文件夹），其中恰好包含一个 `SKILL.md`/`skill.md`。要上传多个技能，请上传多个包。


## 通过 API 创建技能

在文件夹中组装好技能后，通过 API 调用创建该技能。

### 目录上传或 zip 上传


使用 `POST /v1/skills` 上传并验证你的技能，系统会从清单的 frontmatter 中提取名称和描述。你可以上传一个 zip 包，也可以在请求中上传多个文件。

**选项 A：上传文件（multipart）**



```bash
curl --fail-with-body 'https://api.openai.com/v1/skills' \
  -H "Authorization: Bearer $OPENAI_API_KEY" \
  -F 'files[]=@./csv_insights_skill/SKILL.md;filename=csv_insights_skill/SKILL.md;type=text/markdown' \
  -F 'files[]=@./csv_insights_skill/run.py;filename=csv_insights_skill/run.py;type=text/plain' \
  -F 'files[]=@./csv_insights_skill/requirements.txt;filename=csv_insights_skill/requirements.txt;type=text/plain' \
  -F 'files[]=@./csv_insights_skill/assets/example.csv;filename=csv_insights_skill/assets/example.csv;type=text/csv'
```

**选项 B：上传 zip**

```
curl -X POST 'https://api.openai.com/v1/skills' \
  -H "Authorization: Bearer $OPENAI_API_KEY" \
  -F 'files=@./csv_insights_skill.zip;type=application/zip'
```

每个技能使用一种上传方式。zip 能将指令、脚本、依赖和示例输入打包在一起。



**技能对象与版本指针**

响应中包含技能的 `id`、`default_version` 和 `latest_version`。保存 `id` 和一个版本，以便将上传的技能附加到托管 shell。

## 将技能挂载到执行环境

要在 Responses API 中使用技能，请通过 `tools[].environment.skills` 将它们附加到 shell 工具。

### 如何引用技能

- **托管 shell**（`environment.type="container_auto"`）：使用带有 `skill_id` 和可选 `version` 的 `skill_reference`，或者一个 `inline` base64 zip 包。
- **本地 shell**（`environment.type="local"`）：为运行时中可用的文件提供 `name`、`description` 和 `path`。本地 shell 不接受托管的 `skill_reference` 附件。由你的应用执行所请求的命令并返回其输出。

## 可运行示例：`csv_insights_skill` 技能

本演练会在磁盘上创建文件，然后运行 Python 代码片段；它不是一个用 **Run All** 执行的 notebook。请将下面的脚本保存为 `csv_insights_skill/run.py`。

前提条件：Python 3.10 或更高版本、`curl`、`zip`，以及一个 `OPENAI_API_KEY` 环境变量，对应的项目需要能够访问 `gpt-6-astra` 和托管 shell。使用 `python -m pip install --upgrade openai` 安装最新的 Python SDK。

上传命令会在你的 API 项目中创建一个技能。Responses API 调用会产生[模型和容器费用](https://developers.openai.com/api/docs/pricing)。除非你设置 `RUN_SKILLS_API=1`，否则 Python API 示例处于禁用状态。请先运行本地检查，准备好发起 API 请求时再启用。

**1）创建技能文件夹和示例输入。**

```bash
mkdir -p csv_insights_skill/assets
```

完成后的文件夹应包含：

```text
csv_insights_skill/
├── SKILL.md
├── requirements.txt
├── run.py
└── assets/
    └── example.csv
```

将这些依赖保存为 `csv_insights_skill/requirements.txt`。pandas 的 `to_markdown()` 方法需要 `tabulate`。

```text
pandas
matplotlib
tabulate
```

将这份合成输入保存为 `csv_insights_skill/assets/example.csv`：

```csv
quantity,price,category
2,10,books
3,15,games
,20,books
```


**2）创建你的 `SKILL.md`**

```
---
name: csv-insights
description: Summarize a CSV, compute basic stats, and produce a markdown report + a plot image.
---

# CSV Insights Skill

## When to use this
Use this skill when the user provides a CSV file and wants:
- a quick summary (row/col counts, missing values)
- basic numeric statistics
- a simple visualization
- results packaged into an output folder (or zip)

## Inputs
- A CSV file path (local) or a file mounted in the container.

## Outputs
- `output/report.md`
- `output/plot.png` (when the CSV has a numeric column)

## How to run

Run from the directory containing this SKILL.md. The environment must have
pandas, matplotlib, and tabulate installed. If a dependency is unavailable,
report it rather than attempting an unapproved network install.

python run.py --input assets/example.csv --outdir output

Use the output directory requested by the user when one is provided.
Check that report.md exists and that plot.png exists for numeric input.

```

**3）创建你的 `run.py`**

In [ ]:
import argparse
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


def write_report(df: pd.DataFrame, outpath: Path) -> None:
    lines = []
    lines.append(f"# CSV Insights Report\n")
    lines.append(f"**Rows:** {len(df)}  \n**Columns:** {len(df.columns)}\n")
    lines.append("\n## Columns\n")
    lines.append("\n".join([f"- `{c}` ({df[c].dtype})" for c in df.columns]))

    missing = df.isna().sum()
    if missing.any():
        lines.append("\n## Missing values\n")
        for col, count in missing[missing > 0].items():
            lines.append(f"- `{col}`: {int(count)}")
    else:
        lines.append("\n## Missing values\nNo missing values detected.\n")

    numeric = df.select_dtypes(include="number")
    if not numeric.empty:
        lines.append("\n## Numeric summary (describe)\n")
        lines.append(numeric.describe().to_markdown())

    outpath.write_text("\n".join(lines), encoding="utf-8")


def make_plot(df: pd.DataFrame, outpath: Path) -> None:
    numeric = df.select_dtypes(include="number")
    if numeric.empty:
        # No numeric columns → skip plotting
        return

    # Plot the first numeric column as a simple histogram
    col = numeric.columns[0]
    plt.figure()
    df[col].dropna().hist(bins=30)
    plt.title(f"Histogram: {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(outpath)
    plt.close()


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", required=True, help="Path to input CSV")
    parser.add_argument("--outdir", required=True, help="Directory for outputs")
    args = parser.parse_args()

    inpath = Path(args.input)
    outdir = Path(args.outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(inpath)

    write_report(df, outdir / "report.md")
    make_plot(df, outdir / "plot.png")


if __name__ == "__main__":
    main()


**4）先在本地测试，然后将技能打包为 zip。**

在包含 `csv_insights_skill` 的目录中，安装依赖并运行脚本：

```bash
python -m pip install -r csv_insights_skill/requirements.txt
python csv_insights_skill/run.py --input csv_insights_skill/assets/example.csv --outdir local-output
```

`local-output/report.md` 应报告三行、三列，以及一个缺失的 `quantity` 值。`local-output/plot.png` 应包含 `quantity` 的直方图。

只打包技能文件，不要打包生成的输出：

```bash
zip -r csv_insights_skill.zip csv_insights_skill/SKILL.md csv_insights_skill/requirements.txt csv_insights_skill/run.py csv_insights_skill/assets
```

**5）上传技能（实时 API 请求）。**

只有在准备好创建技能时才运行此命令。它会将响应保存到 `skill.json`，供下一步使用。

```bash
curl --fail-with-body 'https://api.openai.com/v1/skills' \
  -H "Authorization: Bearer $OPENAI_API_KEY" \
  -F 'files=@./csv_insights_skill.zip;type=application/zip' \
  -o skill.json
```

**6）通过托管 shell 运行技能（实时 API 请求）。**

设置 `RUN_SKILLS_API=1`，然后在 `skill.json` 所在的同一目录中运行此代码片段。它会固定所上传技能的默认版本，并分析技能包内附带的 CSV。无需单独上传 CSV。

托管 shell 使用其已安装的依赖；出站网络访问默认处于禁用状态。技能会报告缺失的依赖，而不是自动启用网络访问。请将可下载的产物保存在 `/mnt/data` 下，具体参见 [Shell 指南](https://developers.openai.com/api/docs/guides/tools-shell#hosted-runtime-details)。

In [ ]:
import json
import os
from pathlib import Path

from openai import OpenAI

if os.environ.get("RUN_SKILLS_API") == "1":
    client = OpenAI()
    skill = json.loads(Path("skill.json").read_text())
    response = client.responses.create(
        model="gpt-6-astra",
        tools=[
            {
                "type": "shell",
                "environment": {
                    "type": "container_auto",
                    "skills": [
                        {
                            "type": "skill_reference",
                            "skill_id": skill["id"],
                            "version": str(skill["default_version"]),
                        }
                    ],
                },
            }
        ],
        input=(
            "Use the csv-insights skill to analyze its bundled assets/example.csv. "
            "Run the skill's run.py and write outputs to /mnt/data/csv-insights-output. "
            "Verify the row count and missing values, and link to report.md and plot.png."
        ),
    )
    print(response.output_text)
else:
    print("Skipped live API request. Set RUN_SKILLS_API=1 to run.")


**7）请求本地 shell 调用（可选，实时 API 请求）。**

本地 shell 使用你在上面创建的文件，无需上传或引用托管技能。请在执行环境中包含 `csv_insights_skill` 的目录下运行此代码片段。

此代码片段只会请求并显示命令，不会执行它们，也不会完成分析。要完成整个工作流，你的应用必须在沙盒中审查并执行 `shell_call` 命令，捕获 stdout、stderr 和退出结果，并返回带有匹配 `call_id` 的 `shell_call_output`。持续这一过程，直到模型返回最终答案。有关执行循环和输出格式，请参阅 [Local shell mode](https://developers.openai.com/api/docs/guides/tools-shell#local-shell-mode)。

In [ ]:
import os
from pathlib import Path

from openai import OpenAI

if os.environ.get("RUN_SKILLS_API") == "1":
    client = OpenAI()
    skill_path = Path("csv_insights_skill").resolve()
    response = client.responses.create(
        model="gpt-6-astra",
        tools=[
            {
                "type": "shell",
                "environment": {
                    "type": "local",
                    "skills": [
                        {
                            "name": "csv-insights",
                            "description": (
                                "Summarize a CSV, compute basic stats, and produce "
                                "a markdown report + a plot image."
                            ),
                            "path": str(skill_path),
                        }
                    ],
                },
            }
        ],
        input=(
            "Use the csv-insights skill to analyze its bundled assets/example.csv "
            "and write outputs to local-output."
        ),
    )
    for item in response.output:
        if item.type == "shell_call":
            print(item.action.commands)
    print(response.output_text)
else:
    print("Skipped live API request. Set RUN_SKILLS_API=1 to run.")


## 运维最佳实践

**1）让技能"易于发现"**

* 在 frontmatter 中写入**清晰的** `name` 和 `description`。
* 在 `SKILL.md` 中包含：何时使用、如何运行、预期输出、注意事项。

- 添加明确的路由指引："在……时使用"与"在……时不要使用"，以及一些关键的边界情况，全部写在 `SKILL.md` 中。

- 除了正面示例外，还要包含反面示例（*不*应触发该技能的情况），以提升路由准确性。

- 如果路由表现不一致，先迭代名称、描述和示例，再考虑修改代码。

这一点在"批量上传"的讨论中曾被提到：名称和描述应来自 frontmatter，而且应该先用少量技能进行测试。

**2）优先使用 zip 上传，以获得可靠性和可复现性**

* zip 便于移植、易于进行版本管理，并且在上传出现异常时是一个有用的变通办法。

**3）在生产环境中固定版本**

你希望能够说"运行这个版本的流程"，而不是"运行最新的那个，不管是什么"。上传的技能有 **default_version** 和 **latest_version** 两个指针。使用 `POST /v1/skills/{skill_id}/versions` 创建新版本；参见[版本管理与维护](https://developers.openai.com/api/docs/guides/tools-skills#versioning-and-management)。

* 如何固定：`version: "2"`
* 如何浮动：`version: "latest"`
* 省略时会怎样：默认使用 `default_version`

考虑将模型和技能版本一起固定，以便在各个部署之间获得可复现的行为。

**4）像设计小型 CLI 一样设计技能**

一个好的技能脚本：

* 可以从命令行运行
* 输出确定性的 stdout
* 出错时明确报错，并给出用法/错误信息
* 在需要时将输出写入已知的文件路径

在技能中添加具体的模板和完整示例（输入 → 命令 → 预期输出）；模型在调用技能时会读取这些细节，而用于发现的元数据仍然是输入上下文的一部分。当示例与特定工作流相关时，优先将示例和模板放在技能中，而不是使用系统级的少样本提示。

**5）避免在系统提示中重复技能内容**

如果系统提示重复了整个流程，人们就会：

* 绕过技能
* 把逻辑塞进工具 schema

这样你就失去了技能的全部意义（可复用性 + 版本管理 + 按条件调用）。请将系统提示的内容与技能分开。

**6）网络访问**

将技能与开放的网络访问结合使用风险很高。如果必须使用网络访问，请使用严格的允许列表，并将工具输出视为不可信内容。对于用户期望有确认控制的面向消费者的应用，请避免这种配置。
**如果需要网络访问，请将允许列表与明确的"允许哪些数据外传"指引结合使用。**

**7）使用能够可靠执行多步骤工作流的模型**

当模型擅长长上下文推理和多步骤工具执行（文件系统导航、CLI 运行、验证）时，技能的效果最好。
如果你看到任务只完成了一部分或执行不稳定，请升级模型或简化工作流，并在 `SKILL.md` 中添加明确的验证步骤和输出检查。

**限制与验证**

- `SKILL.md` 的匹配不区分大小写
- 只允许恰好一个清单文件（`skill.md`/`SKILL.md`）
- frontmatter 验证遵循 Agent Skills 规范（name 字段）
- zip 上传大小上限：50 MB
- 每个技能版本的文件数量上限：500
- 单个文件解压后大小上限：25 MB

## 总结

技能是提示与工具之间缺失的"中间层"：**提示**定义始终生效的行为，**工具**提供原子能力和副作用，而**技能**则打包可重复的流程（指令 + 脚本 + 资源），让模型**只在需要时挂载并执行。**

**使用技能来保持系统提示精简、工作流持久。** 从小处着手——将一个稳定的流程与清晰的 `SKILL.md` 打包在一起，让它可以像小型 CLI 一样运行，然后发布它。投入生产后，固定版本以确保可复现性，通过发布新版本安全地迭代，并像对待内部标准库一样对待你的技能库：经过审计、易于发现，并在各个智能体之间共享。

随着用户从单轮助手扩展到长时间运行的智能体，技能有助于将"提示面条代码"转变为**可维护、可测试、有版本管理的工作流**——帮助你构建值得信赖、可复用并能随时间演进的智能体行为。

要开始使用 Skills，请查看我们的[文档](https://developers.openai.com/api/docs/guides/tools-skills)。